In [ ]:
# pip install requirments file

# Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from pyDOE3 import lhs
from random import choice
from random import uniform
from numpy.random import randint
from scipy.stats import gmean
from typing import Dict, Any, Tuple, List
import math
import random
import copy
import time
import warnings
import os
import tensorflow as tf
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use GPU 0
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU enabled:", gpus)
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found, running on CPU")

# Data

In [ ]:
with tf.device("GPU:0"):
    df = pd.read_csv('/kaggle/input/ree-syn-30/synthetic_new_30.csv')
    m5 = df[['SiO2', 'Al2O3', 'Fe2O3', 'MgO', 'CaO', 'Na2O', 'K2O', 'TiO2', 'P2O5', 'MnO', 'LOI', 'Total_REE']]
print(m5.head())
print(m5.info())

In [ ]:
# Identify columns that are NOT the target and are numerical after encoding
cols_to_scale = m5.select_dtypes(include=np.number).columns.tolist() 

# Calculate Mean and Standard Deviation (std) for each column
for col in cols_to_scale:
    mean_val = m5[col].mean()
    std_val = m5[col].std()    
    # Apply the Z-score formula: Z = (X - mean) / std
    if std_val != 0:
        m5[col] = (m5[col] - mean_val) / std_val

print(m5.corr())

In [ ]:
# Define the final features (X) and the target (y)
X = m5.drop(columns=['Total_REE']) 
y = m5['Total_REE'] 

print("\n--- FINAL PREPARED DATA SUMMARY ---")
print(f"Final Feature Matrix (X) Shape: {X.shape}")
print(f"Target Vector (y) Shape: {y.shape}")
print("\nFirst 5 Rows of Processed Features (X):")
X.head()

In [ ]:
# Split Data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.5, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.30, random_state=42
)

print(f"Reshaped Training Data Shape for CNN: {X_train.shape}")
print(f"Reshaped Validation Data Shape for CNN: {X_val.shape}")
print(f"Reshaped Test Data Shape for CNN: {X_test.shape}")

# ANN Model

In [ ]:
def build_ann_model(
    input_dim,
    n_layers=2,
    units=64,
    activation="relu",
    dropout=0.2,
    optimizer="adam",
    batch_size=32,
    ep = 100,
    lr=0.001
):
    with tf.device('/GPU:0'):
        model = models.Sequential()
        model.add(layers.Input(shape=(input_dim,)))
    
        for _ in range(n_layers):
            model.add(layers.Dense(units, activation=activation))
            model.add(layers.Dropout(dropout))
    
        model.add(layers.Dense(1, activation="linear"))
    
        if optimizer == "adam":
            opt = optimizers.Adam(learning_rate=lr)
        elif optimizer == "rmsprop":
            opt = optimizers.RMSprop(learning_rate=lr)
        else:
            opt = optimizers.Adam(learning_rate=lr)
    
        model.compile(
            optimizer=opt,
            loss="mse",
            metrics=["mae"]
        )
    with tf.device("/GPU:0"):
        
        early_stop = callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=ep,
            batch_size=batch_size,
            callbacks=[early_stop],
            verbose=1
        )

    return model, history

## Model Hyper-parameters

In [ ]:
ann_hp = {
    "n_layers": {"type": "int", "low": 2, "high": 6},
    "units": {"type": "choice", "choices": [32, 64, 128, 256]},
    "activation": {"type": "choice", "choices": ["relu", "elu"]},
    "dropout_rate": {"type": "float", "low": 0.1, "high": 0.3},
    "learning_rate": {"type": "float", "low": 1e-4, "high": 1e-3},
    "batch_size": {"type": "choice", "choices": [32, 64]},
    "optimizer": {"type": "choice", "choices": ["adam", "rmsprop"]},
    "ep": {"type": "int", "low": 20, "high": 30},
}

## LLM GA process for 

### Initialization of population using lhs

In [ ]:
def generate_population_doe(hp, pop_size=20):
    param_keys = list(hp.keys())
    n_params = len(param_keys)

    # correlation needs multiple samples
    lhs_matrix = lhs(
        n_params,
        samples=pop_size,
        criterion="correlation",
        iterations=50
    )

    population = []
    for row in lhs_matrix:
        candidate = {}
        for i, param in enumerate(param_keys):
            info = hp[param]
            x = row[i]

            if info["type"] == "float":
                candidate[param] = info["low"] + x * (info["high"] - info["low"])

            elif info["type"] == "int":
                candidate[param] = int(
                    round(info["low"] + x * (info["high"] - info["low"]))
                )

            elif info["type"] == "choice":
                idx = min(int(x * len(info["choices"])), len(info["choices"]) - 1)
                candidate[param] = info["choices"][idx]
        
        # candidate = repair_chromosome(candidate)
        population.append(candidate)

    return population

### Selection, crossover, mutation process

In [ ]:
def crossover(parent1, parent2):
    
    child1 = {}
    child2 = {}
    
    child1["n_layers"] = choice([parent1["n_layers"], parent2["n_layers"]])
    child2["n_layers"] = choice([parent1["n_layers"], parent2["n_layers"]])
    
    child1["activation"] = parent2["activation"]
    child2["activation"] = parent1["activation"]
    
    child1["dropout_rate"] = choice([parent2["dropout_rate"], parent1["dropout_rate"]])
    child2["dropout_rate"] = choice([parent1["dropout_rate"], parent2["dropout_rate"]])

    child1["learning_rate"] = choice([parent1["learning_rate"], parent2["learning_rate"]])
    child2["learning_rate"] = choice([parent1["learning_rate"], parent2["learning_rate"]])

    child1["batch_size"] = choice([parent1["batch_size"], parent2["batch_size"]])
    child2["batch_size"] = choice([parent1["batch_size"], parent2["batch_size"]])
    
    child1["optimizer"] = parent2["optimizer"]
    child2["optimizer"] = parent1["optimizer"]

    child1["units"] = parent1["units"]
    child2["units"] = parent2["units"]
    
    child1["ep"] = parent1["ep"]
    child2["ep"] = parent2["ep"]
    return [child1, child2]

In [ ]:
API_KEY =  "Your_API_KEY"
def get_llm():
    return ChatGroq(
        temperature=0,
        groq_api_key=API_KEY,
        model_name="llama-3.3-70b-versatile"
    )


In [ ]:
def crossover_prompt(parent1, parent2):
    return f"""
You are an expert in Genetic Algorithms.

Task:
Perform a SEMANTIC CROSSOVER between two ANN chromosomes and generate TWO valid child chromosomes.

Parent 1:
{parent1}

Parent 2:
{parent2}

Crossover Rules:
- Each child must inherit genes from BOTH parents
- Do NOT copy any parent entirely
- Prefer small, logical combinations of parent traits
- Preserve architectural feasibility
- Ensure all values remain valid and within bounds

Valid Constraints:
- n_layers: integer in [2, 6]
- units: one of {{32, 64, 128, 256}}
- activation: one of {{relu, elu}}
- dropout_rate: float in [0.1, 0.3]
- learning_rate: float in [1e-4, 1e-3]
- batch_size: one of {{32, 64}}
- optimizer: one of {{adam, rmsprop}}
- ep: integer in [50, 100]

Output Requirements:
- Return EXACTLY two child chromosomes
- Output MUST be a JSON LIST: [child1, child2]
- Each child must be a JSON dictionary
- NO explanation, NO markdown, NO extra text
"""
def llm_guided_crossover(parent1, parent2):
    llm = get_llm()
    prompt = crossover_prompt(parent1, parent2)

    response = llm.invoke(prompt)
    parser = JsonOutputParser()
    child = parser.parse(response.content)

    return child


In [ ]:
def prompt(chromosome, p = 0.4):
    """
    Generate a natural language prompt for LLM-based mutation in GA
    """
    prompt = (
        f"I am running hyperparameter tuning of a 1D CNN model using a Genetic Algorithm. "
        f"I will provide you with a chromosome obtained after the crossover operation. "
        f"Your role is to act as an expert in Genetic Algorithms and perform a MUTATION operation.\n\n"
        
        f"Chromosome:\n{chromosome}\n\n"
        
        f"Mutation Rules:\n"
        f"- Mutate ONLY 1 or 2 or 3 genes with some probability {p} (small mutation).\n"
        f"- DO NOT RANDOMIZE THE ENTIRE CHROMOSOME.\n"
        f"- Preserve all unchanged genes exactly.\n"
        f"- Ensure all mutated values remain valid and realistic.\n"
        f"- Prefer incremental changes that improve generalization and robustness.\n\n"
        
        f"Valid Gene Constraints:\n"
        f"- n_layers: integers in [2, 6]\n"
        f"- dropout_rate: floats in [0.1, 0.3]\n"
        f"- activation: one of {{relu, elu}}\n"
        f"- batch_size: one of {{32, 64}}\n"
        f"- learning_rate: floats in [1e-4, 1e-3]\n"
        f"- units (dense units): one of {{32, 64, 128, 256}}\n"
        f"- ep (epochs): integers in [50, 100]\n"
        f"- optimizer: one of {{adam, rmsprop}}n\n"
        
        f"Task:\n"
        f"- Perform mutation according to the rules above.\n"
        f"- Return ONLY the mutated chromosome as a valid JSON dictionary.\n"
        f"- Return the chromosome in jeson format (NO PREAMBLE)\n"
    )
    return prompt

def llm_guided_mutationr(chromosome):
    llm = ChatGroq(
    temperature = 0,
    groq_api_key = API_KEY,
    model_name = 'llama-3.3-70b-versatile'  
    )
    chat = prompt(chromosome)
    response = llm.invoke(chat)
    json_parser = JsonOutputParser()
    json_res = json_parser.parse(response.content)
    return json_res

In [ ]:
def selection(pop_fit):
    pop_fit = np.nan_to_num(pop_fit, nan=0.0, posinf=0.0, neginf=0.0)
    total = np.sum(pop_fit)

    # EDGE CASE: if all fitness values are zero
    if total == 0:
        # fallback: uniform random selection
        indices = list(range(len(pop_fit)))
        return [choice(indices), choice(indices)]

    # Compute probabilities without rounding loss
    probabilities = pop_fit / total

    # Build roulette wheel safely
    selection_wheel = []
    for i, p in enumerate(probabilities):
        count = int(p * 100)  # scale to avoid zero effect
        selection_wheel.extend([i] * max(1, count))  # ensure at least 1 entry

    return [choice(selection_wheel), choice(selection_wheel)]

In [ ]:
def selection_prompt(population, fitness_scores):
    return f"""
You are optimizing ANN hyperparameters using a Genetic Algorithm.

Population:
{population}

Fitness Scores (lower is better):
{fitness_scores}

Task:
- Select TWO parents
- Prefer better fitness
- Maintain diversity
- Avoid selecting identical chromosomes

Output:
- Return a JSON list of exactly TWO selected chromosomes
- No explanation, JSON only
"""
def llm_guided_selection(population, fitness_scores):
    llm = get_llm()
    prompt = selection_prompt(population, fitness_scores)

    response = llm.invoke(prompt)
    parser = JsonOutputParser()
    parents = parser.parse(response.content)

    return parents[0], parents[1]


### Fitness function

In [ ]:
def fitness_cal(model, X_val, y_val, history):
    """
    Normalized RMSE + overfitting penalty
    """

    # ---- SAFE TYPE CONVERSION ----
    if hasattr(y_val, "values"):   
        y_true = y_val.values.astype("float32").ravel()
    else:                           
        y_true = y_val.astype("float32").ravel()

    # ---- PREDICTION ----
    y_pred = model.predict(X_val, verbose=0).ravel()

    # ---- RANGE SAFETY ----
    y_range = max(1e-6, np.max(y_true) - np.min(y_true))

    # ---- NRMSE ----
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    nrmse = rmse / y_range

    # ---- OVERFITTING PENALTY ----
    train_loss = np.mean(history.history["loss"][-3:])
    val_loss   = np.mean(history.history["val_loss"][-3:])
    overfit_gap = max(0.0, val_loss - train_loss)

    # ---- FINAL FITNESS (MINIMIZE) ----
    fitness = nrmse + 0.5 * overfit_gap

    return float(fitness)


### Main code

In [ ]:
generations = 10
threshold = 0.01
num_pop = 20

pop = generate_population_doe(hp=ann_hp,pop_size=num_pop)
acc_best1 = []
par_total1 = []
acc_total1 = []
par_best1 = []
for i, x in enumerate(pop):
    print(f"{i+1} population is: {x}")

In [ ]:
# If you need only LLM-Based mutation
with tf.device("/GPU:0"):

    for generation in range(generations):

        pop_fitness = []

        print("#################")
        print(f"# Generation: {generation+1} #")
        print("#################")

        # ---- FITNESS EVALUATION ----
        for chromosome in pop:

            try:
                model, his = build_ann_model(
                    input_dim=X_train.shape[1],
                    n_layers=chromosome["n_layers"],
                    units=chromosome["units"],
                    activation=chromosome["activation"],
                    dropout=chromosome["dropout_rate"],
                    optimizer=chromosome["optimizer"],
                    batch_size=chromosome["batch_size"],
                    ep=chromosome["ep"],
                    lr=chromosome["learning_rate"]
                )

                acc = fitness_cal(
                    model,
                    X_val,
                    y_val, his
                )

            except Exception as e:
                acc = 0.99
                print("Invalid ANN configuration:", e)

            pop_fitness.append(acc)
            par_total1.append(chromosome)
            acc_total1.append(acc)

            print("Parameters:", chromosome)
            print("Score:", round(acc, 4))

        # ---- ELITISM (BEST INDIVIDUAL) ----
        best_idx = np.argmin(pop_fitness)
        best_fitness = pop_fitness[best_idx]
        best_chromosome = pop[best_idx]

        par_best1.append(best_chromosome)
        acc_best1.append(best_fitness)

        print(f"Min score in generation {generation+1}: {best_fitness:.4f}")
        print(f"Best chromosome in generation {generation+1}: {best_chromosome}")

        if best_fitness < threshold:
            print("Obtained desired score:", best_fitness)
            break

        # ---- PARENT SELECTION ----
        parents_ind = selection(pop_fitness)
        parent1 = pop[parents_ind[0]]
        parent2 = pop[parents_ind[1]]

        # ---- OFFSPRING ----
        children = crossover(parent1, parent2)
        child1 = llm_guided_mutationr(children[0])
        child2 = llm_guided_mutationr(children[1])

        # ---- REMOVE TWO WORST ----
        worst_indices = np.argsort(pop_fitness)[-2:]
        for idx in sorted(worst_indices, reverse=True):
            del pop[idx]

        # ---- ADD CHILDREN ----
        pop.append(child1)
        pop.append(child2)


In [ ]:
# If you need LLM based selection to mution all
with tf.device("/GPU:0"):

    for generation in range(generations):

        pop_fitness = []

        print("#################")
        print(f"# Generation: {generation+1} #")
        print("#################")

        # ---- FITNESS EVALUATION ----
        for chromosome in pop:

            try:
                model, his = build_ann_model(
                    input_dim=X_train.shape[1],
                    n_layers=chromosome["n_layers"],
                    units=chromosome["units"],
                    activation=chromosome["activation"],
                    dropout=chromosome["dropout_rate"],
                    optimizer=chromosome["optimizer"],
                    batch_size=chromosome["batch_size"],
                    ep=chromosome["ep"],
                    lr=chromosome["learning_rate"]
                )

                acc = fitness_cal(
                    model,
                    X_val,
                    y_val, his
                )

            except Exception as e:
                acc = 0.99
                print("Invalid ANN configuration:", e)

            pop_fitness.append(acc)
            par_total1.append(chromosome)
            acc_total1.append(acc)

            print("Parameters:", chromosome)
            print("Score:", round(acc, 4))

        # ---- ELITISM (BEST INDIVIDUAL) ----
        best_idx = np.argmin(pop_fitness)
        best_fitness = pop_fitness[best_idx]
        best_chromosome = pop[best_idx]

        par_best1.append(best_chromosome)
        acc_best1.append(best_fitness)

        print(f"Min score in generation {generation+1}: {best_fitness:.4f}")
        print(f"Best chromosome in generation {generation+1}: {best_chromosome}")

        if best_fitness < threshold:
            print("Obtained desired score:", best_fitness)
            break

        # ---- PARENT SELECTION ----
        parent1, parent2 = llm_guided_selection(pop, pop_fitness)

        # ---- OFFSPRING ----
        children = llm_guided_crossover(parent1, parent2)
        child1 = llm_guided_mutationr(children[0])
        child2 = llm_guided_mutationr(children[1])

        # ---- REMOVE TWO WORST ----
        worst_indices = np.argsort(pop_fitness)[-2:]
        for idx in sorted(worst_indices, reverse=True):
            del pop[idx]

        # ---- ADD CHILDREN ----
        pop.append(child1)
        pop.append(child2)

In [ ]:
for i, x in enumerate(par_total1):
    print(f"Item {i} and hp: {x}")

In [ ]:
for i, x in enumerate(acc_total1):
    print(f"Item {i} and Loss: {x}")

In [ ]:
for i, x in enumerate(par_best1):
    print(f"Generation {i+1} and \nBest hp: {x}")

In [ ]:
for i, x in enumerate(acc_best1):
    print(f"Genertion {i+1} and Best Loss: {x}")

In [ ]:
df = pd.DataFrame({"Hp_set": par_best1, "Loss": acc_best1})
df.sort_values(by='Loss')
# take the best one as hp

# CNN Model

## Reshape the data

In [ ]:
# Reshape Data for 1D CNN
sequence_length = X_train.shape[1]

X_train_reshaped = X_train.values.reshape(
    X_train.shape[0], sequence_length, 1
).astype('float32')
X_val_reshaped = X_val.values.reshape(
    X_val.shape[0], sequence_length, 1
).astype('float32')
X_test_reshaped = X_test.values.reshape(
    X_test.shape[0], sequence_length, 1
).astype('float32')

print(f"Reshaped Training Data Shape for CNN: {X_train_reshaped.shape}")
print(f"Reshaped Validation Data Shape for CNN: {X_val_reshaped.shape}")
print(f"Reshaped Test Data Shape for CNN: {X_test_reshaped.shape}")

## Model

In [ ]:
def build_cnn_model(tar=1, #n_targets,
                    sequence_length=sequence_length,
                    X_train_reshaped=X_train_reshaped,
                    y_train=y_train,
                    X_val_reshaped=X_val_reshaped,
                    y_val=y_val,
                    f1=64,
                    f2=64,
                    k=3,
                    a1='relu',
                    a2='relu',
                    a3='relu',
                    padding='same',
                    p=2, ep=50,
                    op='adam', u=50, batch_size=32):

    with tf.device('/GPU:0'):  
        model = keras.Sequential([
            layers.Conv1D(
                filters=f1,
                kernel_size=k,
                activation=a1,
                padding=padding,
                input_shape=(sequence_length, 1)
            ),
            layers.MaxPooling1D(pool_size=p),

            layers.Conv1D(
                filters=f2,
                kernel_size=k,
                activation=a2,
                padding=padding
            ),
            layers.MaxPooling1D(pool_size=p),

            layers.Flatten(),

            layers.Dense(
                units=u,
                activation=a3,
                kernel_regularizer=regularizers.l2(0.001)
            ),

            layers.Dense(units=tar, activation='linear') #
        ])

        model.compile(
            optimizer=op,
            loss='mse',
            metrics=['mae', 'mse']
        )

    print("\n ---------")

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    with tf.device('/GPU:0'): 
        history = model.fit(
            X_train_reshaped,
            y_train.values.astype('float32'),
            epochs=ep,
            batch_size=batch_size,
            # validation_split=0.1,
            validation_data=(X_val_reshaped, y_val.values.astype("float32")),
            callbacks=[early_stop],
            verbose=1
        )

    return model, history

## Initaialization of hyperparameter

In [ ]:
hp_space = {
    "f1": {"type": "choice", "choices": [16, 32, 64]},
    "f2": {"type": "choice", "choices": [16, 32, 64]},
    "k": {"type": "int", "low": 2, "high": 5},
    "a1": {"type": "choice", "choices": ["relu", "tanh"]},
    "a2": {"type": "choice", "choices": ["relu", "tanh"]},
    "a3": {"type": "choice", "choices": ["relu", "tanh"]},
    "p": {"type": "int", "low": 2, "high": 4},
    "u": {"type": "choice", "choices": [32, 64]},
    "ep": {"type": "int", "low": 50, "high": 100},
    "op": {"type": "choice", "choices": ["adam", "adagrad", "adadelta"]},
}
def generate_population_doe(hp, pop_size=20):
    param_keys = list(hp.keys())
    n_params = len(param_keys)

    # correlation needs multiple samples
    lhs_matrix = lhs(
        n_params,
        samples=pop_size,
        criterion="correlation",
        iterations=50
    )

    population = []
    for row in lhs_matrix:
        candidate = {}
        for i, param in enumerate(param_keys):
            info = hp[param]
            x = row[i]

            if info["type"] == "float":
                candidate[param] = info["low"] + x * (info["high"] - info["low"])

            elif info["type"] == "int":
                candidate[param] = int(
                    round(info["low"] + x * (info["high"] - info["low"]))
                )

            elif info["type"] == "choice":
                idx = min(int(x * len(info["choices"])), len(info["choices"]) - 1)
                candidate[param] = info["choices"][idx]
        
        # candidate = repair_chromosome(candidate)
        population.append(candidate)

    return population

## LLM GA

### Crossover, selection, mutation

In [ ]:
def selection(pop_fit):
    pop_fit = np.nan_to_num(pop_fit, nan=0.0, posinf=0.0, neginf=0.0)
    total = np.sum(pop_fit)

    # EDGE CASE: if all fitness values are zero
    if total == 0:
        # fallback: uniform random selection
        indices = list(range(len(pop_fit)))
        return [choice(indices), choice(indices)]

    # Compute probabilities without rounding loss
    probabilities = pop_fit / total

    # Build roulette wheel safely
    selection_wheel = []
    for i, p in enumerate(probabilities):
        count = int(p * 100)  # scale to avoid zero effect
        selection_wheel.extend([i] * max(1, count))  # ensure at least 1 entry

    return [choice(selection_wheel), choice(selection_wheel)]

In [ ]:
def crossover(parent1, parent2):
    
    child1 = {}
    child2 = {}
    
    child1["f1"] = choice([parent1["f1"], parent2["f1"]])
    child1["f2"] = choice([parent1["f2"], parent2["f2"]])
    
    child2["f1"] = choice([parent1["f1"], parent2["f1"]])
    child2["f2"] = choice([parent1["f2"], parent2["f2"]])
    
    child1["k"] = choice([parent1["k"], parent2["k"]])
    child2["k"] = choice([parent1["k"], parent2["k"]])
    
    child1["a1"] = parent1["a2"]
    child2["a1"] = parent2["a2"]
    
    child1["a2"] = parent2["a1"]
    child2["a2"] = parent1["a1"]
    
    child1["a3"] = choice([parent2["a3"], parent1["a3"]])
    child2["a3"] = choice([parent1["a3"], parent2["a3"]])

    child1["p"] = choice([parent1["p"], parent2["p"]])
    child2["p"] = choice([parent1["p"], parent2["p"]])
       
    child1["op"] = parent2["op"]
    child2["op"] = parent1["op"]

    child1["u"] = parent1["u"]
    child2["u"] = parent2["u"]
    
    child1["ep"] = parent1["ep"]
    child2["ep"] = parent2["ep"]
    return [child1, child2]

In [ ]:
API_KEY =  "YOUR_API_KEY"

def prompt(chromosome, p = 0.4):
    """
    Generate a natural language prompt for LLM-based mutation in GA
    """
    prompt = (
        f"I am running hyperparameter tuning of a 1D CNN model using a Genetic Algorithm. "
        f"I will provide you with a chromosome obtained after the crossover operation. "
        f"Your role is to act as an expert in Genetic Algorithms and perform a MUTATION operation.\n\n"
        
        f"Chromosome:\n{chromosome}\n\n"
        
        f"Mutation Rules:\n"
        f"- Mutate ONLY 1 or 2 or 3 genes with some probability {p} (small mutation).\n"
        f"- DO NOT RANDOMIZE THE ENTIRE CHROMOSOME.\n"
        f"- Preserve all unchanged genes exactly.\n"
        f"- Ensure all mutated values remain valid and realistic.\n"
        f"- Kernel must be <= effective length and Pooling cannot collapse sequence.\n"
        f"- Kernel size must be less than or equal to the input length.\n" 
        f"- Pooling size must not exceed the effective length.\n"
        f"- Kernel size and pooling size must both be at least 1.\n"
        f"- Prefer incremental changes that improve generalization and robustness.\n\n"
        
        f"Valid Gene Constraints:\n"
        f"- f1, f2 (filters): integers in [16, 128], powers of 2 preferred\n"
        f"- k (kernel size): integers in [2, 5]\n"
        f"- a1, a2, a3 (activation): one of {{relu, tanh}}\n"
        f"- p (pool size): integers in [2, 4]\n"
        f"- u (dense units): one of {{32, 64, 128}}\n"
        f"- ep (epochs): integers in [50, 100]\n"
        f"- op (optimizer): one of {{adam, adagrad, adadelta}}n\n"
        
        f"Task:\n"
        f"- Perform mutation according to the rules above.\n"
        f"- Return ONLY the mutated chromosome as a valid JSON dictionary.\n"
        f"- Return the chromosome in jeson format (NO PREAMBLE)\n"
    )
    return prompt

def llm_guided_mutationr(chromosome):
    llm = ChatGroq(
    temperature = 0,
    groq_api_key = API_KEY,
    model_name = 'llama-3.3-70b-versatile'   # 1st we used 'llama-3.1-70b-versatile',
    )
    chat = prompt(chromosome)
    response = llm.invoke(chat)
    json_parser = JsonOutputParser()
    json_res = json_parser.parse(response.content)
    return json_res

In [ ]:
# Repair
def repair_chromosome(chromosome, input_length):
    k = chromosome["k"]
    p = chromosome["p"]

    if k >= input_length:
        k = max(1, input_length // 2)

    eff_len = input_length - 2 * (k - 1)
    eff_len = max(1, eff_len)

    if p >= eff_len:
        p = max(1, eff_len // 2)

    chromosome["k"] = max(1, k)
    chromosome["p"] = max(1, p)

    return chromosome

## Fitness function

In [ ]:
def fitness_cal(model, X_val, y_val, history): 
    y_pred = model.predict(X_val, verbose=0).flatten()
    y_true = y_val.flatten()
    
    # Avoid zero-range normalization
    y_range = max(1e-6, np.max(y_true) - np.min(y_true))

    # NRMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    nrmse = rmse / y_range
    train_loss = np.mean(history.history['loss'][-3:])
    val_loss = np.mean(history.history['val_loss'][-3:])
    overfit_gap = max(0.0, val_loss - train_loss)
    # Final weighted fitness score 
    fitness = nrmse + 0.9 * overfit_gap

    return fitness

## CODE RUN

In [ ]:
generations = 10
threshold = 0.02
num_pop = 20

population = generate_population_doe(hp=hp_space,pop_size=num_pop)
acc_best = []
par_total = []
acc_total = []
par_best = []
for i, x in enumerate(population):
    print(f"{i+1} population is: {x}")

In [ ]:
with tf.device('/GPU:0'):

    for generation in range(generations):

        population_fitness = []

        print("#################")
        print(f"# Generation: {generation+1} #")
        print("#################")

        # ---- FITNESS EVALUATION ----
        for chromosome in population:
            chromosome = repair_chromosome(chromosome, input_length=X_train_reshaped.shape[1])

            try:
                model, his = build_cnn_model(
                    f1=chromosome["f1"],
                    f2=chromosome["f2"],
                    k=chromosome["k"],
                    a1=chromosome["a1"],
                    a2=chromosome["a2"],
                    a3=chromosome["a3"],
                    p=chromosome["p"],
                    op=chromosome["op"],
                    u=chromosome["u"],
                    ep=chromosome["ep"]
                )

                acc = fitness_cal(
                    model,
                    X_val_reshaped,
                    y_val.values.astype("float32"),
                    his
                )

            except Exception as e:
                acc = 0.99
                print("Invalid architecture:", e)

            population_fitness.append(acc)
            par_total.append(chromosome)
            acc_total.append(acc)
            print("Parameters:", chromosome)
            print("Score:", round(acc, 4))

        # ---- ELITISM (BEST INDIVIDUAL) ----
        best_idx = np.argmin(population_fitness)
        best_fitness = population_fitness[best_idx]
        best_chromosome = population[best_idx]

        par_best.append(best_chromosome)
        acc_best.append(best_fitness)

        print(f"Min score in generation {generation+1}: {best_fitness:.4f}")
        print(f"Best chromosome in generation {generation+1}: {best_chromosome}")
        if best_fitness < threshold:
            print("Obtained desired score:", best_fitness)
            break

        # ---- PARENT SELECTION ----
        parents_ind = selection(population_fitness)
        parent1 = population[parents_ind[0]]
        parent2 = population[parents_ind[1]]

        # ---- OFFSPRING ----
        children = crossover(parent1, parent2)
        child1 = llm_guided_mutationr(children[0])
        child2 = llm_guided_mutationr(children[1])

        # ---- REMOVE TWO WORST (BEFORE ADDING CHILDREN) ----
        worst_indices = np.argsort(population_fitness)[-2:]
        for idx in sorted(worst_indices, reverse=True):
            del population[idx]

        # ---- ADD CHILDREN ----
        population.append(child1)
        population.append(child2)

In [ ]:
for i, x in enumerate(par_total):
    print(f"Item {i} and hp: {x}")

In [ ]:
for i, x in enumerate(acc_total):
    print(f"Item {i} and Loss: {x}")

In [ ]:
for i, x in enumerate(par_best):
    print(f"Generation {i+1} and \nBest hp: {x}")

In [ ]:
for i, x in enumerate(acc_best):
    print(f"Genertion {i+1} and Best Loss: {x}")

In [ ]:
df = pd.DataFrame({"Hp_set": par_best, "Loss": acc_best})
df.sort_values(by='Loss')
# Take the best among of these